# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

Unit of analysis: one row = one content page, on one day (page-day grain), from
fact_content_daily_performance joined to dim_content.

Time window: mid-panel month, month=2026-03 (March 2026), for iteration and feature-building.

NOT month=2026-06 (the _sample table / final month) -- that's the sealed test window, and using it now would leak outcome information into my label logic before I've even built it honestly.

In [1]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

con.execute(f"""
    CREATE SECRET hf_token (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    );
""")

# quick connectivity check
con.sql("SELECT COUNT(*) FROM 'hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet'").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          104 │
└──────────────┘



## 2. Fields: feature / label / context / excluded

Feature: gsc_impressions, gsc_avg_position, ga4_sessions, ga4_total_engagement_sec, content_age_days- all measured up to and including the scored day, known before any decision.

Label/proxy: a decline signal built from position/trend movement over the window - never fed back in as an input feature.

Context: content_hash_id, client_hash_id -- joining/grouping only, never features.

Excluded: AI-session data (too sparse - 30,177 of 78.8M rows per the data skill), raw query/URL/title fields (not shipped), and any row before a client's ga4_data_start (ga4_data_available = FALSE means "no tracking yet," not "zero traffic" - treating it as a real zero would poison the features).

In [2]:
con.sql("DESCRIBE SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 3. Verify it with queries (grain, counts, missing values, windows)

Three verification queries below: grain check, row count + date span, and an IS TRUE  availability filter. Then a five-feature frame (each with an "available when?" line), and the deliberate leakage trap -- one label-derived column added on purpose, then removed.

In [3]:
#Query 1: grain check
print("GRAIN CHECK (expect 0 rows back):")
con.sql("""
    SELECT content_hash_id, report_date, COUNT(*) c
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id, report_date
    HAVING c > 1
    LIMIT 5
""").show()

#Query 2: row count + date span
print("ROW COUNT + DATE SPAN:")
con.sql("""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").show()

#Query 3: availability, IS TRUE filter
print("AVAILABILITY (IS TRUE):")
con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").show()

#Five features, each knowable at decision time
features_df = con.sql("""
    SELECT
        f.content_hash_id, f.report_date,
        f.gsc_impressions,
        f.gsc_avg_position,
        f.ga4_sessions,
        f.ga4_total_engagement_sec,
        DATE_DIFF('day', d.content_created_date, f.report_date) AS content_age_days
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet') f
    LEFT JOIN read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet') d
        ON f.content_hash_id = d.content_hash_id
    WHERE f.ga4_data_available IS TRUE
    LIMIT 50000
""").df()
print(features_df.shape)
features_df.head()

#THE TRAP: deliberate leak, watch the score jump, then remove it
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

labeled = features_df.dropna(subset=['gsc_impressions','gsc_avg_position']).copy()
labeled['label'] = (labeled['gsc_avg_position'] > labeled['gsc_avg_position'].median()).astype(int)

X_honest = labeled[['gsc_impressions','ga4_sessions','ga4_total_engagement_sec','content_age_days']]
y = labeled['label']
X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42)
honest_model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, honest_model.predict_proba(X_test)[:,1])
print(f"Honest AUC (no leak): {honest_auc:.3f}")

X_leaky = labeled[['gsc_impressions','ga4_sessions','ga4_total_engagement_sec','content_age_days','gsc_avg_position']]
X_train2, X_test2, y_train2, y_test2 = train_test_split(X_leaky, y, test_size=0.3, random_state=42)
leaky_model = LogisticRegression(max_iter=1000).fit(X_train2, y_train2)
leaky_auc = roc_auc_score(y_test2, leaky_model.predict_proba(X_test2)[:,1])
print(f"Leaky AUC (label-derived column snuck in): {leaky_auc:.3f} -- suspiciously close to 1.0")

print(f"\nKeeping the honest number: {honest_auc:.3f}. Deleting the leaked column from features.")

GRAIN CHECK (expect 0 rows back):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬─────────────┬───────┐
│ content_hash_id │ report_date │   c   │
│     varchar     │    date     │ int64 │
├─────────────────┴─────────────┴───────┤
│                0 rows                 │
└───────────────────────────────────────┘

ROW COUNT + DATE SPAN:
┌───────────┬────────────┬────────────┐
│ row_count │  min_date  │  max_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

AVAILABILITY (IS TRUE):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┐
│ total_rows │ ga4_available_rows │
│   int64    │       int128       │
├────────────┼────────────────────┤
│    9841378 │             413966 │
└────────────┴────────────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(50000, 7)
Honest AUC (no leak): 0.643
Leaky AUC (label-derived column snuck in): 1.000 -- suspiciously close to 1.0

Keeping the honest number: 0.643. Deleting the leaked column from features.


## 4. Data limits

This slice covers only March 2026, one month of ~17 available, and only clients with GA4 tracking already live by then - early-history and highly seasonal clients aren't represented, so patterns found here aren't guaranteed to generalize across the full panel. I'll need to re-check this once I move to multi-month modeling.

In [4]:
con.sql("""
    SELECT client_hash_id, MIN(report_date) as first_seen
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY client_hash_id
    ORDER BY first_seen DESC
    LIMIT 10
""").show()

┌─────────────────────────┬────────────┐
│     client_hash_id      │ first_seen │
│         varchar         │    date    │
├─────────────────────────┼────────────┤
│ client_e00b29e582949543 │ 2026-03-23 │
│ client_810019792c9b8efc │ 2026-03-20 │
│ client_f6f0cdf26d03d7bd │ 2026-03-19 │
│ client_86ebc2f12c01f586 │ 2026-03-03 │
│ client_d211cb07b9059bab │ 2026-03-01 │
│ client_a2eeb8899886adde │ 2026-03-01 │
│ client_c182d11e4862a37d │ 2026-03-01 │
│ client_62f4a7e64f5e0096 │ 2026-03-01 │
│ client_fef1a8f436438636 │ 2026-03-01 │
│ client_8ae2bfb5aa1ffa1e │ 2026-03-01 │
├─────────────────────────┴────────────┤
│ 10 rows                    2 columns │
└──────────────────────────────────────┘



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.